# Polaris RE — Asset-Liability Duration Gap (ALM)

A reinsurer that assumes a coinsurance or modified-coinsurance (Modco) block
takes on the **ceded reserve** and buys assets to back it. Whether that block is
economically hedged depends not just on *how much* the assets earn but on *when*
their cash flows arrive relative to the liability's. The first-order measure of
that timing match is the **asset-liability duration gap**:

$$\text{gap} \;=\; D^{\text{mod}}_{\text{assets}} \;-\; D^{\text{mod}}_{\text{liability}}
\quad(\text{years}).$$

A positive gap means the assets re-price faster than the liability as yields
move (asset-heavy interest-rate exposure); a negative gap means the liability is
longer (reinvestment risk — the classic exposure on a long whole-life block
backed by shorter bonds). The **dollar-duration gap** scales each side by its
value and is the hedgeable quantity: the change in surplus per unit change in
yield.

This notebook is the **validation** notebook for the Asset/ALM epic
(`docs/PLAN_asset_alm.md`, Slice 4b-4). It:

1. builds a seasoned whole-life block, cedes 50% on coinsurance, and reports the
   **dual** duration gap (`analytics.alm.dual_duration_gap`) the CLI, REST API,
   Excel workbook, and dashboard all surface;
2. then reconciles the engine against four **closed forms** — the reserve
   run-off identity, the zero-coupon bond duration/convexity, the
   duration-primitive consistency check, and a perfectly-matched zero-gap block —
   so the numbers above are auditable, not asserted on faith.

Everything is self-contained: a synthetic Gompertz-style mortality curve (no
external table files), so the notebook runs anywhere the package imports.

In [ ]:
from datetime import date
from pathlib import Path

import numpy as np

from polaris_re.analytics.alm import (
    dual_duration_gap,
    duration_gap,
    duration_measures,
    reserve_liability_cash_flows,
)
from polaris_re.assumptions.assumption_set import AssumptionSet
from polaris_re.assumptions.lapse import LapseAssumption
from polaris_re.assumptions.mortality import MortalityTable, MortalityTableSource
from polaris_re.core.asset import AssetPortfolio, Bond
from polaris_re.core.inforce import InforceBlock
from polaris_re.core.policy import Policy, ProductType, Sex, SmokerStatus
from polaris_re.core.projection import ProjectionConfig
from polaris_re.products.dispatch import get_product_engine
from polaris_re.reinsurance.coinsurance import CoinsuranceTreaty
from polaris_re.utils.table_io import MortalityTableArray

## 1. A seasoned whole-life block

Flat mortality builds essentially **no** reserve — a level net premium exactly
funds a constant hazard each period, so there is nothing to accumulate. A
whole-life reserve only builds when mortality *rises* with age (the level
premium over-funds the early, cheap years to pre-fund the expensive late ones).
So we use a synthetic **Gompertz-style** increasing curve
$q_x = 0.0004 \cdot 1.09^{(x-18)}$ — low at younger ages, steep at older ages —
which is self-contained (no external table files) yet produces a realistic
reserve.

The block is ten $1\text{M}$ whole-life policies issued 20 years ago (attained
age 60), valued today.

In [ ]:
VALUATION_DATE = date(2026, 1, 1)
ISSUE_DATE = date(2006, 1, 1)  # seasoned 20 years

# Synthetic Gompertz-style increasing mortality (no data-file dependency).
ages = np.arange(18, 121)
qx_curve = np.clip(0.0004 * 1.09 ** (ages - 18), 0.0, 1.0).astype(np.float64)
qx = qx_curve.reshape(-1, 1)
print(f"q@40 = {qx_curve[40 - 18]:.4f}   q@60 = {qx_curve[60 - 18]:.4f}   q@80 = {qx_curve[80 - 18]:.4f}")

tables = {}
for sex in (Sex.MALE, Sex.FEMALE):
    for smoker in (SmokerStatus.SMOKER, SmokerStatus.NON_SMOKER, SmokerStatus.UNKNOWN):
        tables[f"{sex.value}_{smoker.value}"] = MortalityTableArray(
            rates=qx.copy(),
            min_age=18,
            max_age=120,
            select_period=0,
            source_file=Path("synthetic"),
        )

mortality = MortalityTable(
    source=MortalityTableSource.CSO_2001,
    table_name="Synthetic Gompertz",
    min_age=18,
    max_age=120,
    select_period_years=0,
    has_smoker_distinct_rates=False,
    tables=tables,
)
lapse = LapseAssumption.from_duration_table({1: 0.05, 2: 0.05, 3: 0.05, "ultimate": 0.05})
assumptions = AssumptionSet(
    mortality=mortality, lapse=lapse, version="alm-nb", effective_date=VALUATION_DATE
)

policies = [
    Policy(
        policy_id=f"WL{i:03d}",
        issue_age=40,
        attained_age=60,
        sex=Sex.MALE,
        smoker_status=SmokerStatus.NON_SMOKER,
        underwriting_class="PREFERRED",
        face_amount=1_000_000.0,
        annual_premium=18_000.0,
        policy_term=None,
        duration_inforce=240,
        issue_date=ISSUE_DATE,
        valuation_date=VALUATION_DATE,
        product_type=ProductType.WHOLE_LIFE,
    )
    for i in range(10)
]
block = InforceBlock(policies=policies)

config = ProjectionConfig(
    valuation_date=VALUATION_DATE,
    projection_horizon_years=40,
    discount_rate=0.05,
    maintenance_cost_per_policy_per_year=100.0,
)
gross = get_product_engine(inforce=block, assumptions=assumptions, config=config).project()

print(f"{block.n_policies} policies, total face ${block.total_face_amount():,.0f}")
print(f"Opening reserve ${gross.reserve_balance[0]:,.0f}, peak ${gross.reserve_balance.max():,.0f}")

## 2. Cede 50% on coinsurance — the reinsurer inherits the ceded reserve

Under coinsurance the reinsurer takes a proportional share of **every** cash
flow, reserves included (unlike YRT, which cedes no reserve). So a 50% cession
hands the reinsurer half of the held reserve, and the reinsurer's assets must
back exactly that run-off.

In [ ]:
treaty = CoinsuranceTreaty(cession_pct=0.5)
net, ceded = treaty.apply(gross, inforce=block)

ceded_reserve_0 = ceded.reserve_balance[0]
print(f"Cedant-retained (net) opening reserve : ${net.reserve_balance[0]:,.0f}")
print(f"Ceded (reinsurer) opening reserve     : ${ceded_reserve_0:,.0f}")

## 3. The backing asset portfolio

The reinsurer buys a small bond portfolio to back the ceded reserve: a 10-year
and a 20-year semi-annual coupon bond, sized to the ceded opening reserve. The
`AssetPortfolio` exposes the same risk measures the duration gap consumes —
market value, book yield (the IRR of carrying value vs cash flows), and
Macaulay / modified duration / convexity, all on the engine's effective-annual
monthly discounting convention $v = (1+y)^{-1/12}$.

In [ ]:
y = config.discount_rate  # common ALM valuation yield (single yield isolates the timing gap)
R0 = ceded_reserve_0

portfolio = AssetPortfolio(
    bonds=[
        Bond(face_value=round(R0 * 0.6), coupon_rate=0.045, coupon_frequency=2, term_months=120),
        Bond(face_value=round(R0 * 0.4), coupon_rate=0.040, coupon_frequency=2, term_months=240),
    ],
    portfolio_id="REINSURER-BACKING",
)

print(f"Portfolio face         ${portfolio.total_face_value:,.0f}")
print(f"Market value @ {y:.0%}    ${portfolio.market_value(y):,.0f}   (vs ceded reserve ${R0:,.0f})")
book_yield = portfolio.book_yield()
print(f"Book yield (IRR)       {book_yield:.4%}" if book_yield is not None else "Book yield: undefined")
print(f"Macaulay duration      {portfolio.macaulay_duration(y):.2f} yr")
print(f"Modified duration      {portfolio.modified_duration(y):.2f} yr")
print(f"Convexity              {portfolio.convexity(y):.1f}")

## 4. The dual asset-liability duration gap

`dual_duration_gap` measures the **same** assets against **two** reserve
liabilities at one common yield:

* the **reinsurer-view** gap — assets vs the **ceded** reserve run-off (the
  headline for a reinsurer tool), and
* the **cedant-view** gap — assets vs the cedant-retained (`net`) reserve.

The liability stream is the reserve-backed run-off (Option B, ADR-113): its
present value ties to the held reserve, so its modified duration is the
interest-rate sensitivity of the reserve the assets actually fund. This is the
identical compute path the CLI, REST API, Excel workbook, and Streamlit
dashboard surface — the notebook only *reads* it.

In [ ]:
reserve_rate = config.effective_valuation_rate  # the reserve's own valuation rate
dual = dual_duration_gap(portfolio, net, ceded, y, reserve_rate)


def show_side(label: str, gap) -> None:
    if gap is None:
        print(f"{label}: (undefined — reserve ~0 at this yield)")
        return
    print(
        f"{label}:\n"
        f"  asset modified duration     {gap.asset_modified_duration:7.3f} yr  (MV ${gap.asset_market_value:,.0f})\n"
        f"  liability modified duration {gap.liability_modified_duration:7.3f} yr  (PV ${gap.liability_present_value:,.0f})\n"
        f"  duration gap                {gap.duration_gap:7.3f} yr\n"
        f"  dollar-duration gap         ${gap.dollar_duration_gap:,.0f} per unit yield"
    )


show_side("Reinsurer view (ceded reserve — HEADLINE)", dual.reinsurer)
print()
show_side("Cedant view (retained reserve)", dual.cedant)

The reinsurer-side gap is strongly **negative** (the assets are ~9 years short
of the ~18-year whole-life liability): a long-tailed mortality liability backed
by medium-term bonds carries material **reinvestment risk** — if yields fall, the
maturing bonds reinvest at a lower rate while the liability keeps accruing. We
quantify the fix in §9.

## 5. Validation 1 — the reserve run-off ties to the held reserve

The reserve-backed liability stream is the reserve rolled forward with interest
less what is held for next month, $L_t = R_t\,a - R_{t+1}$ with
$a = (1+i)^{1/12}$. Discounted at that same reserve valuation rate it must
**telescope exactly to the opening reserve** $R_0$ — the defining property that
makes its present value the held reserve. We check it to floating-point
tolerance on the ceded side.

In [ ]:
liab_cfs = reserve_liability_cash_flows(ceded, reserve_rate)
v = (1.0 + reserve_rate) ** (-1.0 / 12.0)
periods = np.arange(1, liab_cfs.shape[0] + 1)
pv_runoff = float((liab_cfs * v**periods).sum())

print(f"PV(ceded run-off) @ {reserve_rate:.1%} = ${pv_runoff:,.2f}")
print(f"Opening ceded reserve        = ${ceded_reserve_0:,.2f}")
np.testing.assert_allclose(pv_runoff, ceded_reserve_0, rtol=0, atol=1e-6)
print("OK — the run-off telescopes to the held reserve.")

## 6. Validation 2 — zero-coupon bond closed forms

A zero-coupon bond paying $1$ at month $12N$ is the cleanest duration anchor:
its Macaulay duration is exactly $N$ years, its modified duration is
$N/(1+y)$, and its convexity is $N(N+1)/(1+y)^2$. The asset-side measures are
independent of any liability, so these are exact.

In [ ]:
N = 10
zcb = AssetPortfolio(
    bonds=[Bond(face_value=1_000.0, coupon_rate=0.0, coupon_frequency=1, term_months=12 * N)]
)
exp_mac = float(N)
exp_mod = N / (1.0 + y)
exp_cvx = N * (N + 1) / (1.0 + y) ** 2

print(f"Macaulay : {zcb.macaulay_duration(y):.6f}  (expect {exp_mac:.6f})")
print(f"Modified : {zcb.modified_duration(y):.6f}  (expect {exp_mod:.6f})")
print(f"Convexity: {zcb.convexity(y):.6f}  (expect {exp_cvx:.6f})")
np.testing.assert_allclose(zcb.macaulay_duration(y), exp_mac, rtol=1e-12)
np.testing.assert_allclose(zcb.modified_duration(y), exp_mod, rtol=1e-12)
np.testing.assert_allclose(zcb.convexity(y), exp_cvx, rtol=1e-12)
print("OK — zero-coupon duration and convexity match the closed form.")

## 7. Validation 3 — one duration primitive, not two

`duration_gap` takes the **asset** measures from the portfolio's own
(separately tested) duration API and the **liability** measures from the generic
`duration_measures`. Fed the portfolio's *own* aggregate cash-flow vector,
`duration_measures` must reproduce the portfolio API exactly — the gap "wires"
one closed form, it does not reimplement a second.

In [ ]:
dm = duration_measures(portfolio.cash_flow_vector(), y)
np.testing.assert_allclose(dm.macaulay_duration, portfolio.macaulay_duration(y), rtol=1e-12)
np.testing.assert_allclose(dm.modified_duration, portfolio.modified_duration(y), rtol=1e-12)
np.testing.assert_allclose(dm.present_value, portfolio.market_value(y), rtol=1e-12)
print("OK — duration_measures on the portfolio cash flows == the portfolio's own API.")

## 8. Validation 4 — a perfectly matched block has zero gap

If the liability cash flows equal the assets' own cash flows, the two sides have
identical duration and the gap (and dollar-duration gap) is **exactly zero** —
the immunisation limit.

In [ ]:
matched_bond = Bond(face_value=1_000.0, coupon_rate=0.06, coupon_frequency=2, term_months=60)
matched_pf = AssetPortfolio(bonds=[matched_bond])
matched_liab = matched_pf.cash_flow_vector()  # liability == the asset's own stream
matched = duration_gap(matched_pf, matched_liab, y)

print(f"duration gap        {matched.duration_gap:.3e} yr")
print(f"dollar-duration gap {matched.dollar_duration_gap:.3e}")
np.testing.assert_allclose(matched.duration_gap, 0.0, atol=1e-12)
np.testing.assert_allclose(matched.dollar_duration_gap, 0.0, atol=1e-6)
print("OK — assets that replicate the liability have zero gap.")

## 9. Closing the gap — lengthening the assets

The §4 mismatch is a *too-short asset* problem. Replacing the coupon bonds with
longer zero-coupon strips pushes the asset modified duration up toward the
~18-year liability and shrinks the reinsurer-side gap by roughly an order of
magnitude — the first-order interest-rate hedge.

In [ ]:
immunised = AssetPortfolio(
    bonds=[
        Bond(face_value=round(R0 * 0.55), coupon_rate=0.0, coupon_frequency=1, term_months=276),
        Bond(face_value=round(R0 * 0.45), coupon_rate=0.0, coupon_frequency=1, term_months=180),
    ],
    portfolio_id="REINSURER-IMMUNISED",
)
immunised_gap = dual_duration_gap(immunised, net, ceded, y, reserve_rate).reinsurer
original_gap = dual.reinsurer

print(f"original  reinsurer gap: {original_gap.duration_gap:7.3f} yr")
print(f"immunised reinsurer gap: {immunised_gap.duration_gap:7.3f} yr")
assert abs(immunised_gap.duration_gap) < abs(original_gap.duration_gap)
print("OK — lengthening the assets materially reduces the duration gap.")

## 10. Takeaway

- The **dual duration gap** turns "the reinsurer holds the ceded reserve" into a
  measured, hedgeable number: assets ~9 years short of a ~18-year whole-life
  liability is a large reinvestment exposure, closed to <1 year by lengthening
  the asset side.
- Every figure reconciles to a **closed form** — the reserve run-off ties to the
  held reserve, zero-coupon duration/convexity match textbook formulas, and the
  gap reuses one duration primitive across asset and liability sides.
- The exact `dual_duration_gap` path shown here is what the CLI
  (`deal.asset_portfolio`), the REST `/api/v1/price`, the deal-pricing Excel
  workbook, and the Streamlit dashboard all surface — this notebook is the
  end-to-end validation behind those surfaces.